# Day 4 — Build Your First AI Agent

Today we move from a basic LLM app to an **AI agent**.

A chatbot usually answers in one shot.  
An agent can:
- decide what to do,
- use a tool,
- observe the result,
- and continue until it can answer.

In this notebook, we will build a **minimal tool-using agent** step by step.

## Learning goals

By the end of this notebook, you should be able to:

- explain what makes an AI agent different from a normal chatbot,
- understand the basic agent loop,
- define simple tools in Python,
- build a small agent that chooses and uses a tool,
- trace how the loop works internally.

This notebook covers the **first half** of Day 4.
Tomorrow you can extend it with memory, more tools, and better error handling.

## What is an AI agent?

A simple way to think about it:

**LLM app**
- user asks a question,
- model answers once.

**AI agent**
- user asks a question,
- model decides whether it needs a tool,
- tool runs,
- result comes back,
- model continues,
- final answer is returned.

That repeated cycle is the core idea behind many agent systems.

## The agent loop

A minimal agent often follows this pattern:

1. Read the user request.
2. Decide whether a tool is needed.
3. Call the tool.
4. Observe the result.
5. Repeat if needed.
6. Return the final answer.

Pseudo-flow:

User -> Model -> Tool -> Model -> Final Answer

In [13]:
# Basic imports for today's notebook

import math
import json
from typing import Callable, Dict

## Step 1 — Create tools

Tools are just functions the agent can use.

We will start with two simple tools:
- a calculator,
- a word counter.

These are intentionally small so the agent logic stays easy to understand.

In [14]:
def calculator(expression: str) -> str:
    """
    Evaluate a simple math expression safely.
    Example: '12 * (4 + 3)'
    """
    allowed_names = {
        "abs": abs,
        "round": round,
        "min": min,
        "max": max,
        "pow": pow,
        "sqrt": math.sqrt,
    }

    try:
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return f"Result: {result}"
    except Exception as e:
        return f"Calculator error: {e}"


def word_counter(text: str) -> str:
    """
    Count the number of words in a string.
    """
    words = text.split()
    return f"Word count: {len(words)}"

In [15]:
tools: Dict[str, Callable[[str], str]] = {
    "calculator": calculator,
    "word_counter": word_counter,
}

tools

{'calculator': <function __main__.calculator(expression: str) -> str>,
 'word_counter': <function __main__.word_counter(text: str) -> str>}

## Step 2 — Test the tools directly

Before letting an agent use tools, always test the tools on their own.

In [16]:
print(calculator("12 * (4 + 3)"))
print(word_counter("AI agents can reason and use tools"))

Result: 84
Word count: 7


## Step 3 — Build a tiny rule-based agent

Real agents often use an LLM to decide which tool to call.

For learning, we will first build a very small **rule-based agent**.
This helps us understand the loop before adding a real model.

In [17]:
def choose_tool(user_query: str):
    query = user_query.lower()

    if any(word in query for word in ["calculate", "math", "+", "-", "*", "/", "sum"]):
        return "calculator"

    if any(word in query for word in ["count words", "word count", "how many words"]):
        return "word_counter"

    return None

## Step 4 — Build the agent loop

This is the key idea.

The agent will:
- inspect the request,
- choose a tool if needed,
- run the tool,
- and then produce a final response.

This is a simplified version of the agent loop used in larger frameworks.

In [18]:
def run_agent(user_query: str) -> str:
    print(f"User: {user_query}")

    tool_name = choose_tool(user_query)

    if tool_name is None:
        return "I do not need a tool here. I can answer directly or ask for clarification."

    print(f"Agent decided to use tool: {tool_name}")

    if tool_name == "calculator":
        expression = (
            user_query.lower()
            .replace("calculate", "")
            .replace("math", "")
            .strip()
        )
        tool_result = tools[tool_name](expression)

    elif tool_name == "word_counter":
        text = (
            user_query.lower()
            .replace("count words in", "")
            .replace("word count for", "")
            .replace("how many words in", "")
            .strip()
        )
        tool_result = tools[tool_name](text)

    else:
        tool_result = "Unknown tool."

    print(f"Tool output: {tool_result}")

    final_answer = f"Final answer: {tool_result}"
    return final_answer

In [19]:
response = run_agent("calculate 25 * (2 + 6)")
print(response)

User: calculate 25 * (2 + 6)
Agent decided to use tool: calculator
Tool output: Result: 200
Final answer: Result: 200


In [20]:
response = run_agent("count words in AI agents are useful for multi-step tasks")
print(response)

User: count words in AI agents are useful for multi-step tasks
Agent decided to use tool: calculator
Tool output: Calculator error: invalid syntax (<string>, line 1)
Final answer: Calculator error: invalid syntax (<string>, line 1)


## What just happened?

Our mini-agent followed this pattern:

- received a user request,
- selected a tool,
- passed input to the tool,
- received the tool output,
- turned that into a final answer.

That is the basic structure behind many tool-using agents.

## Step 5 — Add a traceable loop

Many tutorials teach agents as a visible loop because it makes debugging easier.

We will now make the steps explicit:
- Thought
- Action
- Observation
- Final Answer

This is inspired by the common ReAct-style pattern.

In [21]:
def run_agent_with_trace(user_query: str) -> str:
    print("=== AGENT TRACE START ===")
    print(f"User Query: {user_query}")

    tool_name = choose_tool(user_query)

    if tool_name is None:
        print("Thought: I do not need a tool.")
        print("Final Answer: I can answer directly or ask for clarification.")
        print("=== AGENT TRACE END ===")
        return "I can answer directly or ask for clarification."

    print(f"Thought: I should use the {tool_name} tool.")

    if tool_name == "calculator":
        expression = (
            user_query.lower()
            .replace("calculate", "")
            .replace("math", "")
            .strip()
        )
        print(f"Action: calculator('{expression}')")
        observation = calculator(expression)

    elif tool_name == "word_counter":
        text = (
            user_query.lower()
            .replace("count words in", "")
            .replace("word count for", "")
            .replace("how many words in", "")
            .strip()
        )
        print(f"Action: word_counter('{text}')")
        observation = word_counter(text)

    else:
        observation = "Unknown tool."

    print(f"Observation: {observation}")
    final_answer = f"Based on the tool result, the answer is: {observation}"
    print(f"Final Answer: {final_answer}")
    print("=== AGENT TRACE END ===")

    return final_answer

In [22]:
run_agent_with_trace("calculate 144 / 12")

=== AGENT TRACE START ===
User Query: calculate 144 / 12
Thought: I should use the calculator tool.
Action: calculator('144 / 12')
Observation: Result: 12.0
Final Answer: Based on the tool result, the answer is: Result: 12.0
=== AGENT TRACE END ===


'Based on the tool result, the answer is: Result: 12.0'

In [23]:
run_agent_with_trace("how many words in agents use tools to finish tasks")

=== AGENT TRACE START ===
User Query: how many words in agents use tools to finish tasks
Thought: I should use the word_counter tool.
Action: word_counter('agents use tools to finish tasks')
Observation: Word count: 6
Final Answer: Based on the tool result, the answer is: Word count: 6
=== AGENT TRACE END ===


'Based on the tool result, the answer is: Word count: 6'

## Limitations of this version

This is a good teaching version, but it is still very limited.

Current limitations:
- tool selection is rule-based, not model-based,
- input parsing is fragile,
- there is no memory,
- there is no retry or error recovery,
- the agent can only do one simple step at a time.

Tomorrow, you can improve this by adding:
- more tools,
- better parsing,
- short-term memory,
- a real LLM-based decision step.

## Mini exercise

Try one or more of these:

1. Add a new tool called `character_counter`.
2. Update `choose_tool()` so the agent can select it.
3. Test the agent with your own examples.
4. Make the final answer sound more natural.

Suggested challenge:
- Can you make the agent support both word count and character count?

In [24]:
# Your practice space

# Example:
# def character_counter(text: str) -> str:
#     return f"Character count: {len(text)}"

pass

## Recap

Today you built:
- simple Python tools,
- a tool-selection function,
- a minimal agent,
- and a traceable agent loop.

That is the foundation for more advanced agent systems.
Next, you can add memory, multiple tools, and a real LLM to make the agent more flexible.

## Push-ready note

This is a solid **50% Day 4 notebook** because it already teaches the core agent idea:
an agent is not just a model answer, but a loop that can decide, act, observe, and return a result.

## Part 2 — Make the agent more capable

So far, our agent can:
- choose a simple tool,
- use it once,
- and return a result.

Now we will improve it by adding:
- more tools,
- short-term memory,
- multi-step behavior,
- loop limits,
- and a small guardrail.

## Why memory matters

A useful agent often needs some kind of memory.

For now, we will use **short-term memory**:
- store recent user requests,
- store recent agent actions,
- and inspect that history later.

This is a lightweight version of the session memory patterns commonly used in agent systems.

In [25]:
memory = {
    "history": [],
    "tool_usage": []
}

In [26]:
def save_to_memory(role: str, content: str):
    memory["history"].append({"role": role, "content": content})


def save_tool_usage(tool_name: str, tool_input: str, tool_output: str):
    memory["tool_usage"].append({
        "tool": tool_name,
        "input": tool_input,
        "output": tool_output
    })


def show_memory():
    return memory

## Add two more tools

We will now add:
- `character_counter`
- `text_reverser`

These are simple, but they help us demonstrate multi-tool behavior.

In [27]:
def character_counter(text: str) -> str:
    return f"Character count: {len(text)}"


def text_reverser(text: str) -> str:
    return f"Reversed text: {text[::-1]}"

In [28]:
tools["character_counter"] = character_counter
tools["text_reverser"] = text_reverser

list(tools.keys())

['calculator', 'word_counter', 'character_counter', 'text_reverser']

## Update tool selection

Our agent now needs better routing logic.

In [30]:
def choose_tool_v2(user_query: str):
    query = user_query.lower()

    if any(word in query for word in ["calculate", "math", "+", "-", "*", "/", "sum"]):
        return "calculator"

    if any(phrase in query for phrase in ["count words", "word count", "how many words"]):
        return "word_counter"

    if any(phrase in query for phrase in ["count characters", "character count", "how many characters"]):
        return "character_counter"

    if any(phrase in query for phrase in ["reverse text", "reverse this", "reverse"]):
        return "text_reverser"

    return None

## Add a simple guardrail

Agents should not run forever.

We will use:
- a maximum number of steps,
- and a basic input check.

In [31]:
MAX_STEPS = 3

def basic_guardrail(user_query: str) -> bool:
    blocked_words = ["hack", "attack", "steal"]
    lowered = user_query.lower()
    return not any(word in lowered for word in blocked_words)

## Build a better agent

This version:
- stores history,
- stores tool traces,
- uses the new tools,
- checks a guardrail,
- and stops after a small number of steps.

In [32]:
def extract_tool_input(tool_name: str, user_query: str) -> str:
    query = user_query.lower()

    if tool_name == "calculator":
        return query.replace("calculate", "").replace("math", "").strip()

    if tool_name == "word_counter":
        return (
            query.replace("count words in", "")
                 .replace("word count for", "")
                 .replace("how many words in", "")
                 .strip()
        )

    if tool_name == "character_counter":
        return (
            query.replace("count characters in", "")
                 .replace("character count for", "")
                 .replace("how many characters in", "")
                 .strip()
        )

    if tool_name == "text_reverser":
        return (
            query.replace("reverse text", "")
                 .replace("reverse this", "")
                 .replace("reverse", "")
                 .strip()
        )

    return user_query

In [33]:
def run_agent_v2(user_query: str) -> str:
    print("=== AGENT V2 START ===")

    if not basic_guardrail(user_query):
        return "Request blocked by guardrail."

    save_to_memory("user", user_query)

    steps = 0
    final_answer = None

    while steps < MAX_STEPS:
        steps += 1
        print(f"\nStep {steps}")

        tool_name = choose_tool_v2(user_query)

        if tool_name is None:
            final_answer = "I do not need a tool, or I need better instructions."
            save_to_memory("agent", final_answer)
            print(final_answer)
            break

        print(f"Thought: I should use the {tool_name} tool.")
        tool_input = extract_tool_input(tool_name, user_query)
        print(f"Action: {tool_name}('{tool_input}')")

        tool_output = tools[tool_name](tool_input)
        print(f"Observation: {tool_output}")

        save_tool_usage(tool_name, tool_input, tool_output)

        final_answer = f"Final answer: {tool_output}"
        save_to_memory("agent", final_answer)
        print(final_answer)
        break

    if steps >= MAX_STEPS and final_answer is None:
        final_answer = "Stopped because the agent reached the step limit."
        save_to_memory("agent", final_answer)

    print("=== AGENT V2 END ===")
    return final_answer

In [34]:
run_agent_v2("count characters in artificial intelligence")

=== AGENT V2 START ===

Step 1
Thought: I should use the character_counter tool.
Action: character_counter('artificial intelligence')
Observation: Character count: 23
Final answer: Character count: 23
=== AGENT V2 END ===


'Final answer: Character count: 23'

In [35]:
run_agent_v2("reverse this Kanpur is building cool AI projects")

=== AGENT V2 START ===

Step 1
Thought: I should use the text_reverser tool.
Action: text_reverser('kanpur is building cool ai projects')
Observation: Reversed text: stcejorp ia looc gnidliub si rupnak
Final answer: Reversed text: stcejorp ia looc gnidliub si rupnak
=== AGENT V2 END ===


'Final answer: Reversed text: stcejorp ia looc gnidliub si rupnak'

In [36]:
run_agent_v2("calculate (18 + 7) * 3")

=== AGENT V2 START ===

Step 1
Thought: I should use the calculator tool.
Action: calculator('(18 + 7) * 3')
Observation: Result: 75
Final answer: Result: 75
=== AGENT V2 END ===


'Final answer: Result: 75'

## Inspect memory

We can now look at what the agent remembers during this notebook session.

In [37]:
show_memory()

{'history': [{'role': 'user',
   'content': 'count characters in artificial intelligence'},
  {'role': 'agent', 'content': 'Final answer: Character count: 23'},
  {'role': 'user',
   'content': 'reverse this Kanpur is building cool AI projects'},
  {'role': 'agent',
   'content': 'Final answer: Reversed text: stcejorp ia looc gnidliub si rupnak'},
  {'role': 'user', 'content': 'calculate (18 + 7) * 3'},
  {'role': 'agent', 'content': 'Final answer: Result: 75'}],
 'tool_usage': [{'tool': 'character_counter',
   'input': 'artificial intelligence',
   'output': 'Character count: 23'},
  {'tool': 'text_reverser',
   'input': 'kanpur is building cool ai projects',
   'output': 'Reversed text: stcejorp ia looc gnidliub si rupnak'},
  {'tool': 'calculator', 'input': '(18 + 7) * 3', 'output': 'Result: 75'}]}

## Why this is better

This version is still simple, but it is closer to a real agent because it now has:

- multiple tools,
- visible action traces,
- short-term session memory,
- a guardrail,
- and a loop limit.

These are common building blocks in practical agent systems.

## Optional challenge — multi-step planning

Right now, the agent only performs one tool action before answering.

A stronger version would:
1. plan a task,
2. call one tool,
3. use the observation,
4. decide whether another tool is needed,
5. then return the final answer.

You can try that as an extension.

In [38]:
def plan_task(user_query: str):
    query = user_query.lower()

    if "and then" in query:
        return [part.strip() for part in query.split("and then")]

    return [user_query]

In [39]:
plan_task("count words in ai is amazing and then reverse this ai is amazing")

['count words in ai is amazing', 'reverse this ai is amazing']

## Small exercise

Complete this function so the agent can process more than one step.

In [40]:
def run_multi_step_agent(user_query: str):
    tasks = plan_task(user_query)
    outputs = []

    for task in tasks:
        result = run_agent_v2(task)
        outputs.append(result)

    return outputs

In [41]:
run_multi_step_agent("count words in ai is amazing and then reverse this ai is amazing")

=== AGENT V2 START ===

Step 1
Thought: I should use the word_counter tool.
Action: word_counter('ai is amazing')
Observation: Word count: 3
Final answer: Word count: 3
=== AGENT V2 END ===
=== AGENT V2 START ===

Step 1
Thought: I should use the text_reverser tool.
Action: text_reverser('ai is amazing')
Observation: Reversed text: gnizama si ia
Final answer: Reversed text: gnizama si ia
=== AGENT V2 END ===


['Final answer: Word count: 3', 'Final answer: Reversed text: gnizama si ia']

## Final recap

In Day 4, you built an agent that can:
- select tools,
- execute tools,
- trace actions,
- remember recent interactions,
- and apply a basic guardrail.

That is a meaningful step up from a plain LLM app.

## Tomorrow's upgrade ideas

Good next upgrades:
- replace rule-based routing with an LLM,
- add richer memory,
- improve tool input parsing,
- add retry logic,
- add structured outputs,
- and build a small research assistant project.